In [417]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [418]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.feature_selection import f_regression
from scipy.stats import kruskal

from sklearn.linear_model import Ridge, Lasso
from xgboost import XGBRegressor

import itertools




In [ ]:
df = pd.read_csv('./datasets/treino.csv')
df['Idade_Casa'] = df['YrSold'] - df['YearBuilt']
# print(f"Formato original dos dados: {df.shape}")


Formato original dos dados: (1168, 82)


In [420]:
print("--- 1. Linhas Duplicadas ---")
print(f"Total de duplicatas: {df.duplicated().sum()}\n")

print("--- 2. Colunas Completamente Vazias ---")
colunas_vazias = df.columns[df.isnull().all()].tolist()
print(f"Colunas vazias: {colunas_vazias}\n")

print("--- 3. Top Colunas com mais Nulos ---")
nulos = df.isnull().sum()
nulos_percentual = (nulos / len(df)) * 100
df_nulos = pd.DataFrame({'Total Nulos': nulos, '% Nulos': nulos_percentual})
print(df_nulos[df_nulos['Total Nulos'] > 0].sort_values(by='% Nulos', ascending=False).head(81))

--- 1. Linhas Duplicadas ---
Total de duplicatas: 0

--- 2. Colunas Completamente Vazias ---
Colunas vazias: []

--- 3. Top Colunas com mais Nulos ---
              Total Nulos    % Nulos
PoolQC               1162  99.486301
MiscFeature          1122  96.061644
Alley                1094  93.664384
Fence                 935  80.051370
MasVnrType            683  58.476027
FireplaceQu           547  46.832192
LotFrontage           217  18.578767
GarageType             64   5.479452
GarageYrBlt            64   5.479452
GarageFinish           64   5.479452
GarageQual             64   5.479452
GarageCond             64   5.479452
BsmtCond               28   2.397260
BsmtFinType1           28   2.397260
BsmtExposure           28   2.397260
BsmtQual               28   2.397260
BsmtFinType2           28   2.397260
MasVnrArea              6   0.513699
Electrical              1   0.085616


In [ ]:
# 1. Defina as colunas para excluir
COLUNAS_PARA_EXCLUIR = [
    'Id',              # Apenas o identificador, não ajuda a prever o preço
    # 'GarageArea',      # Excluída pois é colinear com GarageCars
    # 'TotRmsAbvGrd',    # Excluída pois é colinear com GrLivArea
    # # '1stFlrSF'         # Excluída pois é colinear com TotalBsmtSF
    # # --- COMBATENDO O VIF INFINITO (Mantendo apenas os Totais) ---
    # # 'TotalBsmtSF',
    # 'BsmtFinSF1',      
    # 'BsmtFinSF2',     
    # 'BsmtUnfSF',       

    # 'BsmtFinSF1',
    # # 'BsmtFinSF2',
    # '1stFlrSF',        
    # '2ndFlrSF',        
    # # 'LowQualFinSF', 
    # 'YrSold' ,   
    # 'YearBuilt',       
    # 'PoolQC',
    # # 'PoolArea',
    # # 'GarageCond',
    # # 'GarageQual',
    # # # 'Exterior1st',
    # 'Exterior2nd'

]


# 2. Preenchimento EXATO para Categóricas onde "NA" tem significado
PREENCHIMENTO_CATEGORICAS_CONHECIDAS = {
    'Alley': 'No alley access',
    'BsmtQual': 'No Basement',
    'BsmtCond': 'No Basement',
    'BsmtExposure': 'No Basement',
    'BsmtFinType1': 'No Basement',
    'BsmtFinType2': 'No Basement',
    'FireplaceQu': 'No Fireplace',
    'GarageType': 'No Garage',
    'GarageFinish': 'No Garage',
    'GarageQual': 'No Garage',
    'GarageCond': 'No Garage',
    # 'GarageYrBlt':'No Garage',
    'PoolQC': 'No Pool',
    'Fence': 'No Fence',
    'MiscFeature': 'None',
    'MasVnrType':'None'
}



In [422]:
df_limpo = df.copy()
colunas_excluir_existentes = [col for col in COLUNAS_PARA_EXCLUIR if col in df_limpo.columns]
df_limpo = df_limpo.drop(columns=colunas_excluir_existentes)

for col, valor_preenchimento in PREENCHIMENTO_CATEGORICAS_CONHECIDAS.items():
    if col in df_limpo.columns:
        df_limpo[col] = df_limpo[col].fillna(valor_preenchimento)






In [423]:
# Verificando o impacto
nulos_antes = df.isnull().sum().sum()
nulos_depois = df_limpo.isnull().sum().sum()

print(f"Total de valores nulos originais: {nulos_antes}")
print(f"Total de valores nulos após este preenchimento: {nulos_depois}")
print(f"Valores nulos resolvidos com segurança: {nulos_antes - nulos_depois}\n")

print("--- Colunas que AINDA possuem valores nulos (para decidirmos o próximo passo) ---")
nulos_restantes = df_limpo.isnull().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0].sort_values(ascending=False)
print(nulos_restantes)

Total de valores nulos originais: 6227
Total de valores nulos após este preenchimento: 288
Valores nulos resolvidos com segurança: 5939

--- Colunas que AINDA possuem valores nulos (para decidirmos o próximo passo) ---


LotFrontage    217
GarageYrBlt     64
MasVnrArea       6
Electrical       1
dtype: int64


In [424]:
ESTRATEGIAS_NUMERICAS_POR_COLUNA = {
    'LotFrontage': 'mediana', # Distância para a rua faz mais sentido ser a mediana dos vizinhos
    'MasVnrArea': 'zero',     # Se a área de alvenaria for nula, provavelmente é zero
    'GarageYrBlt': 'zero'     # Se não tem garagem, ano zero (ou você poderia colocar a mediana)
}

# 4. Estratégias PADRÃO (Para as colunas que você não listou acima)
# Opções Numéricas: 'zero', 'media', 'mediana'
ESTRATEGIA_NUMERICA_PADRAO = 'mediana'

# Opções Categóricas: 'moda', 'ausente' (Tratará nulos aleatórios como Electrical ou MasVnrType)
ESTRATEGIA_CATEGORICA_PADRAO = 'moda'



In [425]:
# plt.figure(figsize=(8, 4))
# sns.scatterplot(x=df_limpo['GrLivArea'], y=df_limpo['SalePrice'])
# plt.title("GrLivArea vs SalePrice (Veja os pontos extremos à direita)")
# plt.show()

In [426]:
condicao_outlier_area = df_limpo['GrLivArea'] > 4000
df_limpo = df_limpo.drop(df_limpo[condicao_outlier_area].index)
# plt.figure(figsize=(8, 4))
# sns.scatterplot(x=df_limpo['GrLivArea'], y=df_limpo['SalePrice'])
# plt.title("GrLivArea vs SalePrice (Veja os pontos extremos à direita)")
# plt.show()

In [427]:
COLUNAS_PARA_APLICAR_IQR = ['LotArea', 'TotalBsmtSF']

for col in COLUNAS_PARA_APLICAR_IQR:
    Q1 = df_limpo[col].quantile(0.25)
    Q3 = df_limpo[col].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    df_limpo = df_limpo[
        (df_limpo[col] >= limite_inferior) & (df_limpo[col] <= limite_superior) | 
        df_limpo[col].isnull()
    ]

print(f"Tamanho do dataset DEPOIS de remover outliers: {df_limpo.shape}")

Tamanho do dataset DEPOIS de remover outliers: (1067, 80)


In [428]:

# Separar numéricas e categóricas (armazenando como lista para reaproveitamento)
todas_numericas = df_limpo.select_dtypes(include=[np.number]).columns.tolist()
todas_categoricas = df_limpo.select_dtypes(exclude=[np.number]).columns.tolist()

# Aplicar nas colunas NUMÉRICAS
for col in todas_numericas:
    if df_limpo[col].isnull().sum() > 0:
        estr = ESTRATEGIAS_NUMERICAS_POR_COLUNA.get(col, ESTRATEGIA_NUMERICA_PADRAO)
        
        if estr == 'zero':
            df_limpo[col] = df_limpo[col].fillna(0)
        elif estr == 'media':
            df_limpo[col] = df_limpo[col].fillna(df_limpo[col].mean())
        elif estr == 'mediana':
            df_limpo[col] = df_limpo[col].fillna(df_limpo[col].median())

# Aplicar nas colunas CATEGÓRICAS restantes
for col in todas_categoricas:
    if df_limpo[col].isnull().sum() > 0:
        if ESTRATEGIA_CATEGORICA_PADRAO == 'moda':
            df_limpo[col] = df_limpo[col].fillna(df_limpo[col].mode()[0])
        elif ESTRATEGIA_CATEGORICA_PADRAO == 'ausente':
            df_limpo[col] = df_limpo[col].fillna('Ausente')

nulos_finais = df_limpo.isnull().sum().sum()

print(f"Total de nulos agora: {nulos_finais}")

if nulos_finais != 0:
    print("Atenção: Ainda restaram alguns valores nulos. Verifique:")
    print(df_limpo.isnull().sum()[df_limpo.isnull().sum() > 0])

Total de nulos agora: 0


In [ ]:
print("⏳ A gerar os datasets para o teste...\n")

# --- Identificação das Colunas ---
target = 'SalePrice'
cols_num = [c for c in todas_numericas if c != target]
cols_cat = todas_categoricas

# --- VERSÃO 1: Com One-Hot Encoding (Dummies) ---
# O pandas get_dummies por padrão mantém as numéricas e transforma as categóricas
df_ohe = pd.get_dummies(df_limpo, columns=cols_cat, drop_first=True)
print(f"✅ Versão 1 (One-Hot Encoding) criada! Formato: {df_ohe.shape}")

# --- VERSÃO 3: Target Encoding ---
df_target = df_limpo.copy()
te = TargetEncoder(target_type='continuous', random_state=42)

# Ajustamos apenas nas categóricas, mas o resultado deve ser aplicado ao dataframe que já contém as numéricas
df_target[cols_cat] = te.fit_transform(df_target[cols_cat], df_target[target])
print(f"✅ Versão 3 (Target Encoding) criada! Formato: {df_target.shape}")

# --- VERSÃO 4: Misto (Estratégia Robusta com ColumnTransformer) ---
cols_ohe_especifico = ['Foundation', 'GarageType', 'SaleCondition', 'MSZoning', 'CentralAir']
cols_ordinal_especifico = ['ExterQual', 'KitchenQual', 'BsmtQual']
# O resto das categóricas vai para Target Encoding
cols_target_restante = [c for c in cols_cat if c not in cols_ohe_especifico and c not in cols_ordinal_especifico]

# Criamos o transformador garantindo que as numéricas passem (passthrough ou scaler)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', cols_num), # Mantém as numéricas como estão (o scaler será aplicado no avaliar_completo)
        ('target', TargetEncoder(target_type='continuous'), cols_target_restante),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'), cols_ohe_especifico),
        ('ord', OrdinalEncoder(), cols_ordinal_especifico)
    ]
)

# Aplicar a transformação
X_misto_array = preprocessor.fit_transform(df_limpo.drop(target, axis=1), df_limpo[target])

# Recuperar nomes das colunas para reconstruir o DataFrame
# Importante: o ColumnTransformer muda a ordem das colunas (num -> target -> ohe -> ord)
nomes_colunas = (cols_num + 
                 cols_target_restante + 
                 preprocessor.named_transformers_['ohe'].get_feature_names_out(cols_ohe_especifico).tolist() + 
                 cols_ordinal_especifico)
''
df_misto_final = pd.DataFrame(X_misto_array, columns=nomes_colunas)
df_misto_final[target] = df_limpo[target].values # Readecionar o alvo
print(f"✅ Versão 4 (Misto) criada! Formato: {df_misto_final.shape}")

⏳ A gerar os datasets para o teste...

✅ Versão 1 (One-Hot Encoding) criada! Formato: (1067, 240)
✅ Versão 3 (Target Encoding) criada! Formato: (1067, 80)
✅ Versão 4 (Misto) criada! Formato: (1067, 95)


In [430]:
# df_final

In [431]:
# # df_teste_stats = df_ordinal.copy()
# df_teste_stats = df_target.copy()

# X_stats = df_teste_stats.drop(columns=['SalePrice'])
# y_stats = df_teste_stats['SalePrice']

# print("="*60)
# print("🔍 1. MATRIZ DE PEARSON (Pares com Colinearidade > 0.75)")
# print("="*60)
# corr_matrix = X_stats.corr(method='pearson').abs()
# upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# alta_colinearidade = [
#     (upper_tri.columns[i], upper_tri.columns[j], upper_tri.iloc[i, j])
#     for i in range(len(upper_tri.columns))
#     for j in range(i+1, len(upper_tri.columns))
#     if upper_tri.iloc[i, j] > 0.75
# ]

# if alta_colinearidade:
#     for var1, var2, corr in sorted(alta_colinearidade, key=lambda x: x[2], reverse=True):
#         print(f"⚠️ {var1:<15} e {var2:<15} | Correlação: {corr:.3f}")
# else:
#     print("✅ Nenhuma alta correlação direta encontrada!")


# print("\n" + "="*60)
# print("📊 2. TESTE VIF - MULTICOLINEARIDADE (Top 15)")
# print("   (Regra geral: VIF > 10 indica problema grave)")
# print("="*60)
# # Adiciona constante para o modelo matemático do VIF funcionar corretamente
# X_vif = add_constant(X_stats) 
# vif_data = pd.DataFrame()
# vif_data["Feature"] = X_vif.columns
# # Ignoramos avisos de divisão por zero caso existam colunas constantes
# import warnings
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
#     vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

# # Removemos a constante do print e ordenamos
# vif_data = vif_data[vif_data['Feature'] != 'const'].sort_values(by="VIF", ascending=False)
# print(vif_data.head(15).to_string(index=False))


# print("\n" + "="*60)
# print("🎯 3. TESTE F - IMPORTÂNCIA LINEAR (Top 10)")
# print("   (Mede impacto direto da coluna no SalePrice)")
# print("="*60)
# f_values, p_values_f = f_regression(X_stats, y_stats)
# df_f_test = pd.DataFrame({'Feature': X_stats.columns, 'F_Score': f_values, 'P_Value': p_values_f})
# df_f_test = df_f_test.sort_values(by='F_Score', ascending=False)
# print(df_f_test.head(10).to_string(index=False))


# print("\n" + "="*60)
# print("🏷️ 4. TESTE H (Kruskal-Wallis) - CATEGÓRICAS (Top 10)")
# print("   (Impacto estatístico das categorias no SalePrice)")
# print("="*60)
# # Para o teste H, usamos os textos originais do df_limpo para agrupar
# colunas_categoricas_originais = df_limpo.select_dtypes(exclude=[np.number]).columns
# h_results = []

# for col in colunas_categoricas_originais:
#     # Agrupa os preços reais pelas categorias (ex: Vizinhanças)
#     grupos = [grupo['SalePrice'].values for nome, grupo in df_limpo.groupby(col)]
#     # O teste exige pelo menos 2 categorias para comparar
#     if len(grupos) > 1:
#         stat, p_value_h = kruskal(*grupos)
#         h_results.append({'Feature': col, 'H_Statistic': stat, 'P_Value': p_value_h})

# df_h_test = pd.DataFrame(h_results).sort_values(by='H_Statistic', ascending=False)
# print(df_h_test.head(10).to_string(index=False))

In [432]:
# # ==========================================
# # 📊 VISUALIZAÇÃO: MATRIZ DE CORRELAÇÃO DE PEARSON
# # ==========================================


# # Calcula a matriz de correlação completa (usando o df_teste_stats que tem o alvo)
# corr_matrix = df_teste_stats.corr(method='pearson')

# print("⏳ A gerar os gráficos de correlação...")

# # ---------------------------------------------------------
# # GRÁFICO 1: Matriz de Correlação Completa
# # ---------------------------------------------------------
# plt.figure(figsize=(24, 20))

# # Criar uma máscara para esconder o triângulo superior (evita informação duplicada)
# mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# # Escolher a paleta de cores (Azul = correlação negativa, Vermelho = correlação positiva)
# cmap = sns.diverging_palette(230, 20, as_cmap=True)

# # Plotar o heatmap completo (sem números dentro, pois são muitas colunas)
# sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
#             square=True, linewidths=.5, cbar_kws={"shrink": .5})

# plt.title('Matriz de Correlação Completa (Todas as Variáveis)', fontsize=24, pad=20)
# plt.tight_layout()
# plt.show()

# # ---------------------------------------------------------
# # GRÁFICO 2: Zoom nas Variáveis mais correlacionadas com 'SalePrice'
# # ---------------------------------------------------------
# plt.figure(figsize=(12, 10))

# # Selecionar as 15 variáveis com maior correlação absoluta com o preço
# k = 15 
# cols_top = corr_matrix.nlargest(k, 'SalePrice')['SalePrice'].index
# cm_top = np.corrcoef(df_teste_stats[cols_top].values.T)

# # Plotar o heatmap focado (agora com os números dentro)
# sns.set(font_scale=1.1)
# hm = sns.heatmap(cm_top, cbar=True, annot=True, square=True, fmt='.2f', 
#                  annot_kws={'size': 12}, yticklabels=cols_top.values, 
#                  xticklabels=cols_top.values, cmap='Reds')

# plt.title(f'Top {k} Variáveis mais Correlacionadas com SalePrice', fontsize=18, pad=20)
# plt.tight_layout()
# plt.show()

In [433]:
modelos = {
    "Regressão Linear": LinearRegression(), # Não possui parâmetros críticos de ajuste.
    
    "Árvore de Decisão": DecisionTreeRegressor(
        max_depth=10,            # Evita que a árvore cresça infinitamente
        min_samples_leaf=5,      # Cada 'folha' deve ter ao menos 5 casas
        random_state=42
    ),
    
    "Floresta Aleatória": RandomForestRegressor(
        n_estimators=300,        # Mais árvores = mais estabilidade (tente 100 a 500)
        max_depth=15,            # Profundidade moderada
        min_samples_split=5,     
        max_features='sqrt',     # Testa apenas uma parte das colunas por árvore (ajuda muito!)
        random_state=42
    ),
    
    "KNN": KNeighborsRegressor(
        n_neighbors=9,           # Teste valores ímpares: 5, 7, 9, 11
        weights="distance",      # Dá mais importância aos vizinhos mais próximos
        metric='manhattan'       # Em muitas dimensões, manhattan costuma ser melhor que euclidean
    ),
    
    "SVM": SVR(
        kernel='rbf',            # O padrão RBF é excelente para relações não-lineares
        C=10,                    # Tente 1.0, 10, 100 (quanto maior, menos erros tolera)
        epsilon=0.1              # Margem de erro onde o modelo não é penalizado
    ),
    
    "Ensemble (GBM)": GradientBoostingRegressor(
        n_estimators=500,        # Como o aprendizado é gradual, precisamos de mais árvores
        learning_rate=0.05,      # Taxa menor exige mais n_estimators (o 'ajuste fino')
        max_depth=4,             
        subsample=0.8,           # Usa apenas 80% dos dados para cada árvore (reduz overfitting)
        random_state=42
    ),
    
    "Ridge Regression": Ridge(
        alpha=15.0               # Se o erro de treino for muito menor que o de teste, aumente o alpha
    ),
    
    "Lasso Regression": Lasso(
        alpha=0.001        # O Lasso é sensível; tente valores entre 0.0001 e 0.011
    ),
    
    "XGBoost": XGBRegressor(
        n_estimators=1000, 
        learning_rate=0.01,      # Aprendizado bem lento e preciso
        max_depth=4,
        colsample_bytree=0.8,    # Porcentagem de colunas usadas por árvore
        subsample=0.8
    )
}



In [434]:
def avaliar_completo_customizado(nome_versao, df_dados):
    # Verificar se o alvo existe no dataframe recebido
    if 'SalePrice' not in df_dados.columns:
        print(f"❌ Erro na {nome_versao}: Coluna 'SalePrice' não encontrada!")
        return

    print(f"\n{'='*120}")
    print(f"{nome_versao}")
    print(f"{'='*120}")
    
    X = df_dados.drop(columns=['SalePrice'])
    y = df_dados['SalePrice']
    
    # Divisão 80% treino e 20% teste
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Escalonamento
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    y_train_log = np.log1p(y_train)
    
    # Cabeçalho focado em RMSE, RMSLE e R² para Treino e Teste
    print(f"{'Modelo':<20} | {'RMSE Treino':<12} | {'RMSE Teste':<12} | {'RMSLE Treino':<12} | {'RMSLE Teste':<12} | {'R² Treino':<9} | {'R² Teste':<8} |")
    print("-" * 120)
    
    for nome, modelo in modelos.items():
        try:
            # TREINO
            modelo.fit(X_train_scaled, y_train_log)
            
            preds_train_log = modelo.predict(X_train_scaled)
            preds_train_reais = np.clip(np.expm1(preds_train_log), a_min=0, a_max=None)
            
            rmse_train = np.sqrt(mean_squared_error(y_train, preds_train_reais))
            rmsle_train = np.sqrt(mean_squared_log_error(y_train, preds_train_reais))
            r2_train = r2_score(y_train, preds_train_reais)
            
            # TESTE
            preds_val_log = modelo.predict(X_val_scaled)
            preds_val_reais = np.clip(np.expm1(preds_val_log), a_min=0, a_max=None)
            
            rmse_val = np.sqrt(mean_squared_error(y_val, preds_val_reais))
            rmsle_val = np.sqrt(mean_squared_log_error(y_val, preds_val_reais))
            r2_val = r2_score(y_val, preds_val_reais)
            
            # Exibir resultados formatados
            print(f"{nome:<20} | ${rmse_train:>10,.0f} | ${rmse_val:>10,.0f} | {rmsle_train:.5f}      | {rmsle_val:.5f}      | {r2_train:.4f}    | {r2_val:.4f}   |")
        except Exception as e:
            print(f"{nome:<20} | Erro: {str(e)[:50]}")

# Rodar para os datasets que você criou
avaliar_completo_customizado("TESTE 1: ONE-HOT ENCODING", df_ohe)
avaliar_completo_customizado("TESTE 2: TARGET ENCODING (ALL CATS)", df_target)
avaliar_completo_customizado("TESTE 3: ESTRATÉGIA MISTA (ORD/OHE/TARGET)", df_misto_final)


TESTE 1: ONE-HOT ENCODING
Modelo               | RMSE Treino  | RMSE Teste   | RMSLE Treino | RMSLE Teste  | R² Treino | R² Teste |
------------------------------------------------------------------------------------------------------------------------
Regressão Linear     | $    15,300 | $    16,582 | 0.08548      | 0.11002      | 0.9495    | 0.9290   |
Árvore de Decisão    | $    17,345 | $    28,617 | 0.09413      | 0.18615      | 0.9351    | 0.7887   |
Floresta Aleatória   | $    14,501 | $    20,575 | 0.07240      | 0.12220      | 0.9547    | 0.8908   |
KNN                  | $         0 | $    25,918 | 0.00000      | 0.15020      | 1.0000    | 0.8267   |
SVM                  | $    14,228 | $    29,178 | 0.07516      | 0.16509      | 0.9563    | 0.7803   |
Ensemble (GBM)       | $     4,009 | $    16,588 | 0.02190      | 0.10769      | 0.9965    | 0.9290   |
Ridge Regression     | $    15,361 | $    15,192 | 0.08718      | 0.10066      | 0.9491    | 0.9404   |
Lasso Regression  